# What in this Notebook

##### datapreprocessing and cleaning 
##### Duplicate data removal and conflict data analysis and creating separate dataframe without conflict texts.
##### cleaning:manual contradiction handling as my data contain don't as don t 
##### url,html tag,user tag, email tag, contradictions, numbers, emojis... removal(my data does not contain any of these).Convert UC to LC
##### stop word removal using spacy.Installed spacy.imported stopwords and manually removed important words from stopword set
#####  no_conflict_data and train_df after stop word removal converted to csv

# import

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import seaborn as sns
import pandas as pd
train_df=pd.read_csv("train_data.csv")
validation_df=pd.read_csv("validation_data.csv")
test_df=pd.read_csv("Testing_data.csv")

# Data Preprocessing And Cleaning

### DUPLICATE ANALYSIS

In [2]:
train_df.drop_duplicates(inplace=True)

In [3]:
print(train_df.duplicated().sum())

0


In [4]:
train_df["text"].isnull().sum()


np.int64(0)

In [5]:
train_df["text"].duplicated().value_counts()

text
False    15969
True        30
Name: count, dtype: int64

In [6]:
duplicate=train_df[train_df["text"].duplicated(keep=False)].sort_values(by="text")
print(duplicate)

                                                    text  label
3508   i am not amazing or great at photography but i...      1
8246   i am not amazing or great at photography but i...      2
5277   i began to feel accepted by gaia on her own terms      2
15705  i began to feel accepted by gaia on her own terms      1
8804   i bet taylor swift basks in the knowledge that...      4
15572  i bet taylor swift basks in the knowledge that...      3
11013  i cant escape the tears of sadness and just tr...      2
12892  i cant escape the tears of sadness and just tr...      1
13653    i could feel myself hit this strange foggy wall      4
14314    i could feel myself hit this strange foggy wall      5
7578   i feel a remembrance of the strange by justin ...      5
11525  i feel a remembrance of the strange by justin ...      4
9071        i feel im being generous with that statement      2
9786        i feel im being generous with that statement      1
8557                              i feel

In [7]:
int(duplicate['text'].value_counts().max())

2

In [8]:
duplicate['label'].value_counts()

label
1    16
2    15
4    11
3     7
5     7
0     4
Name: count, dtype: int64

In [9]:
no_conflict_train_df=train_df.copy()
print(no_conflict_train_df)
print("number of Duplicates:",no_conflict_train_df["text"].duplicated().sum())
if(no_conflict_train_df["text"].duplicated().sum()>0):
 no_conflict_train_df=no_conflict_train_df.drop_duplicates(subset="text",keep=False)


                                                    text  label
0                                i didnt feel humiliated      0
1      i can go from feeling so hopeless to so damned...      0
2       im grabbing a minute to post i feel greedy wrong      3
3      i am ever feeling nostalgic about the fireplac...      2
4                                   i am feeling grouchy      3
...                                                  ...    ...
15995  i just had a very brief time in the beanbag an...      0
15996  i am now turning and i feel pathetic that i am...      0
15997                     i feel strong and good overall      1
15998  i feel like this was such a rude comment and i...      3
15999  i know a lot but i feel so stupid because i ca...      0

[15999 rows x 2 columns]
number of Duplicates: 30


In [10]:
print(no_conflict_train_df.value_counts().sum())
    

15939


In [11]:
no_conflict_train_df.to_csv("no_conflict_duplicate_data.csv")

In [12]:
validation_df["text"].duplicated().sum()

np.int64(2)

In [13]:
validation_df[validation_df["text"].duplicated(keep=False)].sort_values(by="text")

,text,label
774,i feel so tortured by it,4
1993,i feel so tortured by it,3
300,i have had several new members tell me how com...,2
603,i have had several new members tell me how com...,1


In [14]:
validation_df=validation_df.drop_duplicates(subset="text",keep=False)

In [15]:
validation_df.to_csv("no_duplicate_validation.csv",index=False)

### CLEANING

In [16]:
%pip install contractions
%pip install emoji


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
import  re
import contractions
import string
import emoji
print(type(train_df))

<class 'pandas.DataFrame'>


In [18]:
train_df["text"].str.contains("dont",na=False).sum()

np.int64(454)

In [19]:
def contradiction_manual(data):
  change_contradiction={
    "didn t":"did not",
    "didnt":"did not",
    "dont":"do not",
    "don t":"do not",
    "doesn t":"does not",
    "isn t":"is not",
    "aren t":"are not",
    "wasn t":"was not",
    "wasnt":"was not",
    "weren t":"were not",
    "won t":"will not",
    "can t":"cannot",
    "im":"i am",
    "ive":"i have"
    }
  for old,new in change_contradiction.items():
    data=data.str.replace(old,new,regex=True)
  return data

In [20]:
def preprocessing(data):
    url=data.str.contains(r'https?://\S+|www\.\S+',regex=True,na=False).sum()
    print("URL TAGS",url)
    if(url>0):
        data=data.str.replace(r'https?://\S+|www\.\S+','',regex=True)
        print("URL removed")

    html=data.str.contains(r'<.*?>',regex=True,na=False).sum()
    print("HTML TAGS",html)
    if(html>0):
        data=data.str.replace(r'<.*?>','',regex=True)
        print("HTML removed")

    email=data.str.contains(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',regex=True,na=False).sum()
    print("emial TAGS",email)
    if(email>0):
        data=data.str.replace(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}','',regex=True)
        print("Email removed")

    user=data.str.contains(r'@\w+',regex=True,na=False).sum()
    print("@USER TAGS",user)
    if(user>0):
        data=data.str.replace(r'@\w+','',regex=True)
        print("User tags removed")

    upper=data.str.contains(r'[A-Z]',regex=True,na=False).sum()
    print("UPPERCASE",upper)
    if(upper>0):
        data=data.str.lower()
        print("Changed to lowercase")

    xspace=data.str.contains(r'^\s|\s$|\s{2,}|\t|\n',regex=True,na=False).sum()
    print("Extra Space",xspace)
    if(xspace>0):
        data=data.str.split().str.join(" ")
        print("Extra space removed")

    contrac=data.str.contains(r"\b\w+'\w+\b",regex=True,na=False).sum()
    print("Contractions",contrac)
    if(contrac>0):
        data=data.apply(contractions.fix)
    data=contradiction_manual(data)    
    print("contradictions fixed")

    punctuations=data.str.contains(r'[^\w\s]',regex=True,na=False).sum()
    print("punctuations and symbols:",punctuations)
    if(punctuations>0):
       data=data.str.translate(str.maketrans("","",string.punctuation))

    e_count=(data.apply(emoji.emoji_count).sum())
    pattern=r":-\)|:\)|:-\(|:\(|:D|:-D|;\)|;-\)|:P|:-P|:p|:-p|<3|:'\(|:\||:-\|"
    emoticon=data.str.contains(pattern,regex=True,na=False).sum()
    print("emoticons:",emoticon)
    print("emoji:",e_count)
    if(e_count>0 ):
        data=data.apply(lambda x:emoji.replace_emoji(x,""))
        print("emojis removed")
    if(emoticon>0):
        data=data.str.replace(pattern,"",regex=True)
        print("emoticons removed")

    
    Number=data.str.contains(r'\d',regex=True,na=False).sum()
    print("Numbers:",Number)
    if(Number>0):
        data=data.str.replace(r"\d+","",regex=True)

    return data

In [21]:
train_df["text"]=preprocessing(train_df["text"])

URL TAGS 0
HTML TAGS 0
emial TAGS 0
@USER TAGS 0
UPPERCASE 0
Extra Space 0
Contractions 0
contradictions fixed
punctuations and symbols: 0
emoticons: 0
emoji: 0
Numbers: 0


In [22]:
no_conflict_train_df["text"]=preprocessing(no_conflict_train_df["text"])

URL TAGS 0
HTML TAGS 0
emial TAGS 0
@USER TAGS 0
UPPERCASE 0
Extra Space 0
Contractions 0
contradictions fixed
punctuations and symbols: 0
emoticons: 0
emoji: 0
Numbers: 0


In [40]:
validation_df["text"]=preprocessing(validation_df["text"])

URL TAGS 0
HTML TAGS 0
emial TAGS 0
@USER TAGS 0
UPPERCASE 0
Extra Space 0
Contractions 0
contradictions fixed
punctuations and symbols: 0
emoticons: 0
emoji: 0
Numbers: 0


In [41]:
validation_df.to_csv("validation_preprocessed.csv",index=False)

In [42]:
processed_train_df=train_df.copy()
train_df

,text,label
0,not feel humiliated,0
1,can go feeling so hopeless so damned hopeful w...,0
2,grabbing minute post feel greedy wrong,3
3,ever feeling nostalgic fireplace will know sti...,2
4,feeling grouchy,3
...,...,...
15995,very brief ti ame beanbag said anna feel like ...,0
15996,now turning feel pathetic still waiting tables...,0
15997,feel strong good overall,1
15998,feel like such rude comment glad t,3


In [43]:
no_conflict_train_df

,text,label
0,not feel humiliated,0
1,can go feeling so hopeless so damned hopeful w...,0
2,grabbing minute post feel greedy wrong,3
3,ever feeling nostalgic fireplace will know sti...,2
4,feeling grouchy,3
...,...,...
15995,very brief ti ame beanbag said anna feel like ...,0
15996,now turning feel pathetic still waiting tables...,0
15997,feel strong good overall,1
15998,feel like such rude comment glad t,3


# STOP WORD REMOVAL

In [44]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [45]:
!python -m spacy download en_core_web_sm

Traceback (most recent call last):
  File "c:\Users\jeena\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connection.py", line 204, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jeena\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\util\connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jeena\AppData\Local\Programs\Python\Python311\Lib\socket.py", line 961, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno 11001] getaddrinfo failed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\jeena\AppData\Local\Programs\Python\Py

In [46]:
import spacy

In [47]:
nlp=spacy.load("en_core_web_sm")


In [48]:
print(len(nlp.Defaults.stop_words))
original_stopwords=nlp.Defaults.stop_words
stopwords=original_stopwords.copy()
for word in sorted(stopwords):
    print(word)
print(len(stopwords))

326
'd
'll
'm
're
's
've
a
about
above
across
after
afterwards
again
against
all
almost
alone
along
already
also
although
always
am
among
amongst
amount
an
and
another
any
anyhow
anyone
anything
anyway
anywhere
are
around
as
at
back
be
became
because
become
becomes
becoming
been
before
beforehand
behind
being
below
beside
besides
between
beyond
both
bottom
but
by
ca
call
can
cannot
could
did
do
does
doing
done
down
due
during
each
eight
either
eleven
else
elsewhere
empty
enough
even
ever
every
everyone
everything
everywhere
except
few
fifteen
fifty
first
five
for
former
formerly
forty
four
from
front
full
further
get
give
go
had
has
have
he
hence
her
here
hereafter
hereby
herein
hereupon
hers
herself
him
himself
his
how
however
hundred
i
if
in
indeed
into
is
it
its
itself
just
keep
last
latter
latterly
least
less
made
make
many
may
me
meanwhile
might
mine
more
moreover
most
mostly
move
much
must
my
myself
n't
name
namely
neither
never
nevertheless
next
nine
no
nobody
none
noone
nor
not

In [49]:
print(type(stopwords))
print(len(stopwords))
stopwords.difference_update({'again','all','almost','already','although','always',"no","now",'none','not','nor','nothing','please'})
stopwords.difference_update({'because','both','can','cannot','could','each','either','enough','even','ever','except','further'})
stopwords.difference_update({'get','give','go','hence','here','first','full','how','however','if','indeed','keep','least'})
stopwords.difference_update({'less','may','meanwhile','might','more','moreover','most','much','must',"n't",'never','nevertheless'})
stopwords.difference_update({'often','only','other','otherwise','own','perhaps','quite','rather','really','same','see','seem'})
stopwords.difference_update({'seemed','seeming','seems','serious','should','since','so','sometimes','still','such','than'})
stopwords.difference_update({'therefore','though','thus','too','unless','until','very','well','what','whatever','when','where'})
stopwords.difference_update({'whether','which','while','who','whom','whose','why','will','without','would','yes'})
print(len(stopwords))

<class 'set'>
326
231


def remove_stopwords(data):
    doc=nlp.pipe(data)
    cleaned_text=[]
    for docs in doc:
     cleaned_text.append(" ".join(word.text for word in docs if word.text not in stopwords))
    return cleaned_text

In [50]:
def remove_stopwords(data):
    words=data.split()
    cleaned_text=[word for word in words if word not in stopwords]
    return " ".join(cleaned_text)

In [51]:
train_df["text"]=train_df["text"].apply(remove_stopwords)
no_conflict_train_df["text"]=no_conflict_train_df["text"].apply(remove_stopwords)

In [52]:
no_conflict_train_df.head()

,text,label
0,not feel humiliated,0
1,can go feeling so hopeless so damned hopeful w...,0
2,grabbing minute post feel greedy wrong,3
3,ever feeling nostalgic fireplace will know sti...,2
4,feeling grouchy,3


In [53]:
train_df.tail()

,text,label
15995,very brief ti ame beanbag said anna feel like ...,0
15996,now turning feel pathetic still waiting tables...,0
15997,feel strong good overall,1
15998,feel like such rude comment glad t,3
15999,know lot feel so stupid because can not portray,0


In [54]:
train_df.head()

,text,label
0,not feel humiliated,0
1,can go feeling so hopeless so damned hopeful w...,0
2,grabbing minute post feel greedy wrong,3
3,ever feeling nostalgic fireplace will know sti...,2
4,feeling grouchy,3


In [55]:
train_df

,text,label
0,not feel humiliated,0
1,can go feeling so hopeless so damned hopeful w...,0
2,grabbing minute post feel greedy wrong,3
3,ever feeling nostalgic fireplace will know sti...,2
4,feeling grouchy,3
...,...,...
15995,very brief ti ame beanbag said anna feel like ...,0
15996,now turning feel pathetic still waiting tables...,0
15997,feel strong good overall,1
15998,feel like such rude comment glad t,3


In [56]:
train_df["text"].str.len().describe()

count    15999.000000
mean        64.332708
std         36.656762
min          3.000000
25%         36.000000
50%         57.000000
75%         84.000000
max        239.000000
Name: text, dtype: float64

In [57]:
train_df.to_csv("stop_word_removed_dataset.csv",index=False)

In [58]:
no_conflict_train_df.to_csv("no_conflict_duplicate_data.csv")